## Shared functions

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
import matplotlib.patheffects as path_effects

# colors
COLOR_LE = "#2a78d6"   # blue
COLOR_RCS = "#eb6834"  # orange
INK = "#0b0b0b"
MUTED = "#898781"
GRID = "#e1e0d9"


# Colors keyed by real layer name. The "1-layer" (whole retina) group isn't a real layer, so it gets its own
# neutral color.
LAYER_COLORS = {
    "NFL+GCL": "dodgerblue",
    "IPL": "darkorange",
    "Nuclear": "mediumseagreen",
    "IS+OS": "purple",
}
WHOLE_RETINA_COLOR = "gray"
ML_HATCH = "."


def layer_color(layer):
    return LAYER_COLORS.get(layer, WHOLE_RETINA_COLOR)


# LE has 4 resolvable layers (NFL+GCL, IPL, Nuclear, IS+OS); RCS only has the
# 3 that were actually resolved (no IS+OS data for RCS).
SPECIES_LAYERS = {
    "LE": ["NFL+GCL", "IPL", "Nuclear", "IS+OS"],
    "RCS": ["NFL+GCL", "IPL", "Nuclear"],
}

# Display-name overrides, per species (the underlying dict key stays
# "Nuclear" everywhere; in LE it corresponds to INL+OPL+ONL, per the
# notebook's own column-provenance note above).
LAYER_DISPLAY_NAMES = {
    "LE": {"Nuclear": "INL+OPL+ONL"},
	"RCS": {"Nuclear": "INL"},
}

# Extra horizontal gap
LAYER_TO_WHOLE_RETINA_GAP = 0.5


def display_layer_name(species, layer):
    return LAYER_DISPLAY_NAMES.get(species, {}).get(layer, layer)


def prep_decoded_data(species, group_spacing=1.0):
    """
    Returns (categories, res2dinv_vals, ml_vals, edge_colors, x) for one
    species -- resistivity already converted from Ω·cm to kΩ·cm. Layers
    plus a trailing "1-layer" (whole-retina) group; edge_colors is the
    per-position color (real layer -> its fixed color, "1-layer" -> grey).
    group_spacing scales the spacing between x positions before the
    "1-layer" gap is added (only the hatched-column version widens it, to
    leave room for its sideways-shifted labels).
    """
    table = TABLES[species]
    res2dinv_row = table[METHODS["Res2DInv"][species]]
    ml_row = table[METHODS["ML"][species]]
    layers = SPECIES_LAYERS[species]

    categories = [display_layer_name(species, layer) for layer in layers] + ["1-layer"]
    res2dinv_vals = [res2dinv_row[layer] for layer in layers] + \
        [table["1 layer Res2DInv"]["whole retina"]]
    ml_vals = [ml_row[layer] for layer in layers] + \
        [table["1 layer ML"]["whole retina"]]

    # Ω·cm -> kΩ·cm (the trailing show_min/show_max booleans pass through
    # unscaled).
    def to_kohm(val):
        low, mean, high, show_min, show_column, show_max = val
        return (low / 1000, mean / 1000, high / 1000, show_min, show_column, show_max)

    res2dinv_vals = [to_kohm(val) for val in res2dinv_vals]
    ml_vals = [to_kohm(val) for val in ml_vals]

    edge_colors = [layer_color(layer) for layer in layers] + [WHOLE_RETINA_COLOR]

    x = np.arange(len(categories), dtype=float) * group_spacing
    x[-1] += LAYER_TO_WHOLE_RETINA_GAP
    return categories, res2dinv_vals, ml_vals, edge_colors, x


def decoded_ylim_top(res2dinv_vals, ml_vals, step=0.5):
    """Smallest multiple of `step` at or above the highest whisker/band
    edge in either method's values, so the y-axis top tick/gridline is
    never short of the tallest mark."""
    all_highs = [max(low, high) for (low, mean, high, show_min, show_column, show_max) in res2dinv_vals + ml_vals]
    return step * np.ceil(max(all_highs) / step)


def draw_whisker(ax, xc, low, mean, high, width, color=INK, zorder=4,
                  show_min=True, show_max=True):
    """One manually-drawn whisker (not bar()'s yerr, since decoding is
    nonlinear and can put the "minus" decode above the mean -- yerr
    requires non-negative offsets). Drawn as two independent halves, split
    at `mean`: low -> mean (capped at low) and mean -> high (capped at
    high), so each half can be hidden on its own via show_min / show_max."""
    cap = width * 0.22
    lo, hi = min(low, high), max(low, high)
    if show_min:
        ax.plot([xc, xc], [lo, mean], color=color, linewidth=1.3, zorder=zorder)
        ax.plot([xc - cap, xc + cap], [lo, lo], color=color, linewidth=1.3, zorder=zorder)
    if show_max:
        ax.plot([xc, xc], [mean, hi], color=color, linewidth=1.3, zorder=zorder)
        ax.plot([xc - cap, xc + cap], [hi, hi], color=color, linewidth=1.3, zorder=zorder)


def decoded_mean_label(ax, x, y, mean, ha="center"):
    """One mean-value label in the shared style: bold black with a thin
    white outline for legibility over hatching/whiskers."""
    ax.text(x, y, f"{mean:.2f}", ha=ha, va="bottom", fontsize=14,
             color=INK, fontweight="bold", zorder=5,
             path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


def finish_decoded_axes(ax, x, categories, species, legend_handles):
    """Shared axis cosmetics for every decoded-resistivity plot: x tick
    labels/padding, y-axis label and gridlines (major every 0.5 kΩ·cm,
    labeled; minor every 0.25 kΩ·cm, unlabeled), title, legend, and spine
    cleanup."""
    ax.set_xticks(x)
    ax.set_xticklabels(categories, fontsize=18)
    ax.tick_params(axis="x", pad=12)
    ax.set_ylabel("Decoded resistivity (kΩ·cm)", fontsize=20, labelpad=35)
    ax.legend(handles=legend_handles, fontsize=16, frameon=False)
    ax.yaxis.set_major_locator(MultipleLocator(1))
    ax.yaxis.set_minor_locator(MultipleLocator(0.5))
    ax.tick_params(axis="y", which="major", labelsize=18)
    ax.tick_params(axis="y", which="minor", length=0)
    ax.grid(axis="y", which="major", color=GRID, linewidth=0.8, zorder=0)
    ax.grid(axis="y", which="minor", color=GRID, linewidth=0.5, zorder=0)
    ax.set_axisbelow(True)

    for spine in ax.spines.values():
	    spine.set_edgecolor('black')
	    spine.set_linewidth(1)

    plt.tight_layout()


## Source Data: Decoded resistivity (mean ± std) per layer

In [ ]:
# Each value is (mean-std decode, mean decode, mean+std decode, show_min,
# show_column, show_max). show_min / show_max control whether that value's
# lower ("min") / upper ("max") whisker half is drawn; show_column, when
# False, suppresses that whole method+layer position -- bar, whisker, and
# label -- entirely. All True (draw everything) by default.

LE = {
    "4 layers ML": {
        "NFL+GCL": (1142, 3027, 4741, True, True, True),
        "IPL": (1757, 1712, 2011, True, True, True),
        "Nuclear": (1003, 2168, 3070, True, True, True),
        "IS+OS": (621, 939, 1383, True, True, True),
    },
    "4 layers Res2DInv": {
        "NFL+GCL": (1385, 2687, 3784, True, True, True),
        "IPL": (1201, 1626, 1770, True, True, True),
        "Nuclear": (2325, 2254, 3204, True, True, True),
        "IS+OS": (167, 1026, 1341, True, True, True),
    },
    "1 layer ML": {"whole retina": (1309.4, 1960.44, 2605.53, True, True, True)},
    "1 layer Res2DInv": {"whole retina": (956.36, 1685.82, 2270.58, True, True, True)},
}

RCS = {
    "3 layers ML": {
        "NFL+GCL": (2227, 3373, 4433, True, True, True),
        "IPL": (744, 1120, 1538, True, True, True),
        "Nuclear": (2191, 4045, 5876, True, True, True),
    },
    "3 layers Res2DInv": {
        "NFL+GCL": (4340, 4236, 4810, False, True, False),
        "IPL": (728, 1325, 1850, False, True, False),
        "Nuclear": (1021, 1517, 1987, False, True, False),
    },
    "1 layer ML": {"whole retina": (1241, 1970, 2695, True, True, True)},
    "1 layer Res2DInv": {"whole retina": (1179, 1625, 2143, False, True, False)},
}

# method name -> model-row key, per species (the layer-resolving rows only)
METHODS = {
    "ML": {"LE": "4 layers ML", "RCS": "3 layers ML"},
    "Res2DInv": {"LE": "4 layers Res2DInv", "RCS": "3 layers Res2DInv"},
}
COMMON_LAYERS = ["NFL+GCL", "IPL", "Nuclear"]
TABLES = {"LE": LE, "RCS": RCS}


## Plots

In [ ]:
import matplotlib as mpl


def plot_decoded_bars(species, res2dinv_striped=True):
    """
    Grouped bar chart of decoded layer resistivity for one species: each
    resolved layer gets a Res2DInv bar and an ML bar side by side (whiskers
    span the decoded mean-std -> mean+std range), plus a "1-layer" group,
    set off by an extra horizontal gap, on the far right with the two
    1-layer-model bars.
      - Res2DInv columns get a diagonal hatch by default; pass
        res2dinv_striped=False for plain white-filled Res2DInv columns
        instead.
      - ML columns are solid-filled with the layer color (Res2DInv stays
        white-filled with just a colored outline, hatched or not).
      - Mean-value labels sit at the column's own height (not a shared
        bottom line), shifted left (Res2DInv) / right (ML) of the bar
        center so they clear the whisker running up the middle.
      - The gap between groups is widened a bit (via prep_decoded_data's
        group_spacing) to leave room for that horizontal label offset.
      - Each value's trailing show_min/show_max booleans (set in the
        source data tables) control whether that whisker's lower ("min") /
        upper ("max") half is drawn; show_column, when False, hides that
        whole bar (plus its whisker and label) entirely.
    """
    RES2DINV_HATCH = "///" if res2dinv_striped else ""
    old_hatch_linewidth = mpl.rcParams["hatch.linewidth"]
    mpl.rcParams["hatch.linewidth"] = 1.8

    categories, res2dinv_vals, ml_vals, edge_colors, x = prep_decoded_data(species, group_spacing=1.15)
    res2dinv_mean = np.array([v[1] for v in res2dinv_vals], dtype=float)
    ml_mean = np.array([v[1] for v in ml_vals], dtype=float)
    width = 0.38

    fig, ax = plt.subplots(figsize=(12, 6))
    bars_res2dinv = ax.bar(x - width / 2, res2dinv_mean, width,
                            color="white", edgecolor=edge_colors, linewidth=1.4,
                            hatch=RES2DINV_HATCH, label="Res2DInv", zorder=3)
    bars_ml = ax.bar(x + width / 2, ml_mean, width,
                      color=edge_colors, edgecolor=edge_colors, linewidth=1.4,
                      label="ML", zorder=3)

    for rect, (low, mean, high, show_min, show_column, show_max) in list(zip(bars_res2dinv, res2dinv_vals)) + \
            list(zip(bars_ml, ml_vals)):
        if not show_column:
            rect.set_visible(False)
            continue
        draw_whisker(ax, rect.get_x() + rect.get_width() / 2, low, mean, high, width,
                     show_min=show_min, show_max=show_max)

    ax.set_ylim(0, decoded_ylim_top(res2dinv_vals, ml_vals))

    # Labels sit at the column's own height, shifted outward (left for
    # Res2DInv, right for ML) past the whisker caps so they don't overlap
    # the black min/max lines running up the middle of each pair.
    y_lo_lim, y_hi_lim = ax.get_ylim()
    text_pad = 0.015 * (y_hi_lim - y_lo_lim)
    label_shift = width * 0.32

    for rect, mean, (low, orig_mean, high, show_min, show_column, show_max) in \
            zip(bars_res2dinv, res2dinv_mean, res2dinv_vals):
        if not show_column:
            continue
        xc = rect.get_x() + rect.get_width() / 2
        decoded_mean_label(ax, xc - label_shift, mean + text_pad, mean, ha="right")
    for rect, mean, (low, orig_mean, high, show_min, show_column, show_max) in \
            zip(bars_ml, ml_mean, ml_vals):
        if not show_column:
            continue
        xc = rect.get_x() + rect.get_width() / 2
        decoded_mean_label(ax, xc + label_shift, mean + text_pad, mean, ha="left")

    legend_handles = [
        Patch(facecolor="white", edgecolor=INK, linewidth=1.2,
              hatch=(RES2DINV_HATCH * 1 if RES2DINV_HATCH else None), label="Res2DInv"),
        Patch(facecolor=INK, edgecolor=INK, linewidth=1.2, label="ML"),
    ]
    finish_decoded_axes(ax, x, categories, species, legend_handles)
    mpl.rcParams["hatch.linewidth"] = old_hatch_linewidth
    return fig, ax


plot_decoded_bars("LE")
plot_decoded_bars("RCS")